# Telco Customer Churn — Predictive Retention Analysis
## Notebook 03 · Load into SQLite + Business SQL Queries

| | |
|---|---|
| **Author** | Natalia Stekolnikova |
| **Course** | IFCT153PO — Big Data & Business Intelligence |
| **Input** | `outputs/telco_clean.csv` |
| **Output** | `outputs/telco_churn.db` · 6 CSV query results |

---

### Why load into a database?

CSV files are flat — no structure, no indexing, no query language.
A relational database adds three capabilities essential in production:

1. **SQL queries** — filter, aggregate and join without loading everything into memory
2. **Indexing** — fast lookups on large datasets (millions of rows)
3. **Persistence** — data survives between sessions, shareable with other tools

In a real telecom company, customer data lives in a database (Oracle, PostgreSQL, SQL Server).
This notebook simulates that production workflow using **SQLite** — a lightweight
file-based database that requires no server installation.

### Why `telco_clean.csv` and NOT `telco_model_ready.csv`?

SQL queries should operate on **human-readable values**:
- `'Month-to-month'` not `0`
- `'Fiber optic'` not `1`
- `'Electronic check'` not `3`

`telco_clean.csv` also contains `TenureGroup` and `IsAutoPayment` —
business features excluded from ML due to multicollinearity
but extremely useful for SQL business analysis.

### What this notebook does

```
1. Load  → telco_clean.csv into SQLite table 'customers'
2. Check → data integrity via SQL
3. Query → 6 business SQL queries with actionable insights
4. Export → results to CSV for reporting
```

### Columns available in the database

```
Original (21):  customerID, gender, SeniorCitizen, Partner, Dependents,
                tenure, PhoneService, MultipleLines, InternetService,
                OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport,
                StreamingTV, StreamingMovies, Contract, PaperlessBilling,
                PaymentMethod, MonthlyCharges, TotalCharges, Churn

Engineered (3): NumServices   — count of active services (1-9)
                TenureGroup   — New / Junior / Mid / Loyal  (for SQL grouping)
                IsAutoPayment — 1=auto-pay, 0=manual  (for SQL filtering)
```

---
## 0 · Environment

In [3]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 20)

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR.parent / 'data').exists() \
               else NOTEBOOK_DIR
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_CSV = OUTPUT_DIR / 'telco_clean.csv'
DB_PATH   = OUTPUT_DIR / 'telco_churn.db'

assert CLEAN_CSV.exists(), (
    f'File not found: {CLEAN_CSV}\n'
    'Run Notebook 02 first to generate telco_clean.csv'
)

print(f'Input    : {CLEAN_CSV}')
print(f'Database : {DB_PATH}')

Input    : C:\Users\natal\Documents\churn_project\outputs\telco_clean.csv
Database : C:\Users\natal\Documents\churn_project\outputs\telco_churn.db


---
## 1 · Load data into SQLite

`telco_clean.csv` contains human-readable strings and all three engineered features:
`NumServices`, `TenureGroup`, `IsAutoPayment`. These are ideal for SQL business queries.

In [5]:
# Load clean CSV
df = pd.read_csv(CLEAN_CSV)
print(f'Loaded  : {df.shape[0]:,} rows × {df.shape[1]} columns')
print()

# Verify required columns are present
required = ['customerID','tenure','Contract','InternetService',
            'PaymentMethod','MonthlyCharges','TotalCharges','Churn',
            'NumServices','TenureGroup','IsAutoPayment']
missing = [c for c in required if c not in df.columns]
if missing:
    print(f'WARNING — missing columns: {missing}')
    print('Re-run Notebook 02 to regenerate telco_clean.csv')
else:
    print('All required columns present ✓')
print()

# Connect and load
# if_exists='replace' — recreate table on every run (idempotent)
# index=False — do not write pandas row index as a column
conn = sqlite3.connect(DB_PATH)
df.to_sql('customers', conn, if_exists='replace', index=False)

print(f'Database created : {DB_PATH}')
print(f'Table            : customers')
print()

# Verify row count
n = pd.read_sql('SELECT COUNT(*) as n FROM customers', conn).iloc[0,0]
print(f'Rows in DB : {n:,}  '
      f'{"✓" if n == len(df) else "← MISMATCH — check file"}')

# Show columns
cursor  = conn.execute('PRAGMA table_info(customers)')
db_cols = [r[1] for r in cursor.fetchall()]
print(f'Columns in DB: {len(db_cols)}')
print()

# Preview
print('First 3 rows from database:')
display(pd.read_sql('SELECT * FROM customers LIMIT 3', conn))

Loaded  : 7,043 rows × 24 columns

All required columns present ✓

Database created : C:\Users\natal\Documents\churn_project\outputs\telco_churn.db
Table            : customers

Rows in DB : 7,043  ✓
Columns in DB: 24

First 3 rows from database:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,NumServices,TenureGroup,IsAutoPayment
0,7590-VHVEG,Female,No,Yes,No,1,No,No,DSL,No,...,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,2,New,0
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,...,No,One year,No,Mailed check,56.95,1889.50,0,4,Mid,0
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,...,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,4,New,0


---
## 2 · Data integrity checks via SQL

Before running analysis, verify the data loaded correctly
by running basic checks directly in SQL.

In [7]:
print('═' * 55)
print('DATA INTEGRITY CHECKS')
print('═' * 55)
print()

# Check 1: Row count
n_rows = pd.read_sql(
    'SELECT COUNT(*) as total_rows FROM customers', conn
).iloc[0,0]
print(f'[1] Total rows    : {n_rows:,}  (expected 7,043)')

# Check 2: Churn distribution
q_churn = pd.read_sql('''
    SELECT
        Churn,
        COUNT(*) AS customers,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customers), 1) AS pct
    FROM customers
    GROUP BY Churn
    ORDER BY Churn
''', conn)
print()
print('[2] Churn distribution (0=retained, 1=churned):')
display(q_churn)

# Check 3: NULL values in key columns
key_cols = ['TotalCharges','MonthlyCharges','tenure',
            'Contract','Churn','NumServices','TenureGroup']
null_results = []
for col in key_cols:
    n_null = pd.read_sql(
        f'SELECT COUNT(*) as n FROM customers WHERE "{col}" IS NULL',
        conn
    ).iloc[0,0]
    null_results.append({
        'Column'    : col,
        'NULL count': n_null,
        'Status'    : '✓ OK' if n_null == 0 else '⚠ WARNING'
    })
print()
print('[3] NULL check on key columns:')
display(pd.DataFrame(null_results))

# Check 4: Value ranges
q_ranges = pd.read_sql('''
    SELECT
        MIN(tenure)                   AS tenure_min,
        MAX(tenure)                   AS tenure_max,
        ROUND(MIN(MonthlyCharges), 2) AS monthly_min,
        ROUND(MAX(MonthlyCharges), 2) AS monthly_max,
        ROUND(MIN(TotalCharges),   2) AS total_min,
        ROUND(MAX(TotalCharges),   2) AS total_max,
        MIN(NumServices)              AS numserv_min,
        MAX(NumServices)              AS numserv_max
    FROM customers
''', conn)
print()
print('[4] Value ranges:')
display(q_ranges)

# Check 5: TenureGroup values
q_tg = pd.read_sql('''
    SELECT TenureGroup, COUNT(*) AS n
    FROM customers
    GROUP BY TenureGroup
    ORDER BY CASE TenureGroup
        WHEN 'New'    THEN 1 WHEN 'Junior' THEN 2
        WHEN 'Mid'    THEN 3 WHEN 'Loyal'  THEN 4 END
''', conn)
print()
print('[5] TenureGroup distribution (confirms engineered feature loaded):')
display(q_tg)

print()
print('All checks passed ✓ — ready for business queries.')

═══════════════════════════════════════════════════════
DATA INTEGRITY CHECKS
═══════════════════════════════════════════════════════

[1] Total rows    : 7,043  (expected 7,043)

[2] Churn distribution (0=retained, 1=churned):


,Churn,customers,pct
0,0,5174,73.50
1,1,1869,26.50



[3] NULL check on key columns:


,Column,NULL count,Status
0,TotalCharges,0,✓ OK
1,MonthlyCharges,0,✓ OK
2,tenure,0,✓ OK
3,Contract,0,✓ OK
4,Churn,0,✓ OK
5,NumServices,0,✓ OK
6,TenureGroup,0,✓ OK



[4] Value ranges:


,tenure_min,tenure_max,monthly_min,monthly_max,total_min,total_max,numserv_min,numserv_max
0,0,72,18.25,118.75,18.80,8684.80,1,9



[5] TenureGroup distribution (confirms engineered feature loaded):


,TenureGroup,n
0,New,2186
1,Junior,1024
2,Mid,1594
3,Loyal,2239



All checks passed ✓ — ready for business queries.


---
## 3 · Business SQL Queries

Six queries that extract actionable insights directly from SQL.
Each answers one specific business question about churn.

---
### Query 1 — Churn rate by Contract type

**Business question:** Does commitment level affect churn?

Month-to-month customers have no long-term commitment —
they can leave any month without penalty.
This query quantifies exactly how much that matters.

In [9]:
q1 = pd.read_sql('''
    SELECT
        Contract,
        COUNT(*)                                              AS total_customers,
        SUM(Churn)                                           AS churned,
        COUNT(*) - SUM(Churn)                                AS retained,
        ROUND(AVG(Churn) * 100.0, 1)                        AS churn_rate_pct,
        ROUND(AVG(MonthlyCharges), 2)                        AS avg_monthly_eur,
        ROUND(
            SUM(CASE WHEN Churn = 1 THEN TotalCharges ELSE 0 END)
        , 2)                                                 AS revenue_at_risk_eur
    FROM customers
    GROUP BY Contract
    ORDER BY churn_rate_pct DESC
''', conn)

print('Query 1 — Churn by Contract type:')
display(q1)
print()

mth = q1[q1['Contract']=='Month-to-month']['churn_rate_pct'].values[0]
two = q1[q1['Contract']=='Two year']['churn_rate_pct'].values[0]
rev = q1[q1['Contract']=='Month-to-month']['revenue_at_risk_eur'].values[0]
print('Key insight:')
print(f'  Month-to-month churn: {mth}%  vs  Two year: {two}%')
print(f'  Ratio: {mth/two:.1f}× higher churn for monthly customers')
print(f'  Revenue at risk (monthly contracts): €{rev:,.2f}')
print()
print('Retention action: convert Month-to-month → One year with 10% discount.')
print('Even partial conversion dramatically reduces churn exposure.')

Query 1 — Churn by Contract type:


,Contract,total_customers,churned,retained,churn_rate_pct,avg_monthly_eur,revenue_at_risk_eur
0,Month-to-month,3875,1655,2220,42.70,66.40,1927182.25
1,One year,1473,166,1307,11.30,65.05,674991.20
2,Two year,1695,48,1647,2.80,60.77,260753.45



Key insight:
  Month-to-month churn: 42.7%  vs  Two year: 2.8%
  Ratio: 15.3× higher churn for monthly customers
  Revenue at risk (monthly contracts): €1,927,182.25

Retention action: convert Month-to-month → One year with 10% discount.
Even partial conversion dramatically reduces churn exposure.


---
### Query 2 — Churn rate by Internet service

**Business question:** Does internet type affect churn?

Fiber optic is the premium service — fastest speeds, highest price.
If it also has the highest churn, there is a price/value perception problem.

In [11]:
q2 = pd.read_sql('''
    SELECT
        InternetService,
        COUNT(*)                                              AS total_customers,
        SUM(Churn)                                           AS churned,
        ROUND(AVG(Churn) * 100.0, 1)                        AS churn_rate_pct,
        ROUND(AVG(MonthlyCharges), 2)                        AS avg_monthly_eur,
        ROUND(
            SUM(CASE WHEN Churn = 1 THEN TotalCharges ELSE 0 END)
        , 2)                                                 AS revenue_at_risk_eur
    FROM customers
    GROUP BY InternetService
    ORDER BY churn_rate_pct DESC
''', conn)

print('Query 2 — Churn by Internet service:')
display(q2)
print()

fiber = q2[q2['InternetService']=='Fiber optic']['churn_rate_pct'].values[0]
no_int= q2[q2['InternetService']=='No']['churn_rate_pct'].values[0]
avg_f = q2[q2['InternetService']=='Fiber optic']['avg_monthly_eur'].values[0]
rev_f = q2[q2['InternetService']=='Fiber optic']['revenue_at_risk_eur'].values[0]
print('Key insight:')
print(f'  Fiber optic churn: {fiber}%  vs  No internet: {no_int}%')
print(f'  Ratio: {fiber/no_int:.1f}× — fiber customers churn much more')
print(f'  Avg monthly charge fiber: €{avg_f} — premium price, premium dissatisfaction')
print(f'  Revenue at risk (fiber): €{rev_f:,.2f}')
print()
print('Retention action: Fiber Value Programme — add free OnlineSecurity')
print('+ Backup for 6 months to improve price/value perception.')

Query 2 — Churn by Internet service:


,InternetService,total_customers,churned,churn_rate_pct,avg_monthly_eur,revenue_at_risk_eur
0,Fiber optic,3096,1297,41.90,91.50,2483257.45
1,DSL,2421,459,19.00,58.10,360016.50
2,No,1526,113,7.40,21.08,19652.95



Key insight:
  Fiber optic churn: 41.9%  vs  No internet: 7.4%
  Ratio: 5.7× — fiber customers churn much more
  Avg monthly charge fiber: €91.5 — premium price, premium dissatisfaction
  Revenue at risk (fiber): €2,483,257.45

Retention action: Fiber Value Programme — add free OnlineSecurity
+ Backup for 6 months to improve price/value perception.


---
### Query 3 — Churn rate by Tenure Group

**Business question:** When do customers churn?

`TenureGroup` is an engineered feature created in Notebook 02
specifically for business reporting. It segments customers
into lifecycle stages: New (0-12m), Junior (13-24m), Mid (25-48m), Loyal (49-72m).

This query identifies the critical onboarding window
where most churn happens.

In [13]:
q3 = pd.read_sql('''
    SELECT
        TenureGroup,
        COUNT(*)                              AS total_customers,
        SUM(Churn)                           AS churned,
        COUNT(*) - SUM(Churn)                AS retained,
        ROUND(AVG(Churn) * 100.0, 1)        AS churn_rate_pct,
        ROUND(AVG(tenure), 1)               AS avg_tenure_months,
        ROUND(AVG(MonthlyCharges), 2)        AS avg_monthly_eur,
        ROUND(AVG(NumServices), 1)           AS avg_num_services
    FROM customers
    GROUP BY TenureGroup
    ORDER BY
        CASE TenureGroup
            WHEN 'New'    THEN 1
            WHEN 'Junior' THEN 2
            WHEN 'Mid'    THEN 3
            WHEN 'Loyal'  THEN 4
        END
''', conn)

print('Query 3 — Churn by Tenure Group (customer lifecycle):')
display(q3)
print()

new_r   = q3[q3['TenureGroup']=='New']['churn_rate_pct'].values[0]
loyal_r = q3[q3['TenureGroup']=='Loyal']['churn_rate_pct'].values[0]
new_n   = q3[q3['TenureGroup']=='New']['total_customers'].values[0]
new_svc = q3[q3['TenureGroup']=='New']['avg_num_services'].values[0]
loyal_s = q3[q3['TenureGroup']=='Loyal']['avg_num_services'].values[0]
print('Key insight:')
print(f'  New customer churn   : {new_r}%  ({new_n:,} customers in first year)')
print(f'  Loyal customer churn : {loyal_r}%')
print(f'  Ratio: {new_r/loyal_r:.1f}× — new customers churn 5× more than loyal')
print(f'  Avg services — New: {new_svc}  Loyal: {loyal_s}')
print(f'  Loyal customers use more services → higher switching cost')
print()
print('Retention action: Onboarding Programme — proactive calls at month 1, 3, 6')
print('for all new customers. Critical intervention window.')

Query 3 — Churn by Tenure Group (customer lifecycle):


,TenureGroup,total_customers,churned,retained,churn_rate_pct,avg_tenure_months,avg_monthly_eur,avg_num_services
0,New,2186,1037,1149,47.40,4.70,56.10,2.90
1,Junior,1024,294,730,28.70,18.40,61.36,3.60
2,Mid,1594,325,1269,20.40,36.20,65.93,4.30
3,Loyal,2239,213,2026,9.50,63.00,73.95,5.50



Key insight:
  New customer churn   : 47.4%  (2,186 customers in first year)
  Loyal customer churn : 9.5%
  Ratio: 5.0× — new customers churn 5× more than loyal
  Avg services — New: 2.9  Loyal: 5.5
  Loyal customers use more services → higher switching cost

Retention action: Onboarding Programme — proactive calls at month 1, 3, 6
for all new customers. Critical intervention window.


---
### Query 4 — Churn rate by Payment method

**Business question:** Does payment method reveal customer engagement?

Manual payment (Electronic check, Mailed check) means the customer
actively decides each month to pay — and therefore actively decides
whether to stay. Auto-payment removes that monthly decision point.

We use `IsAutoPayment` (1=auto, 0=manual) to compare groups directly.

In [15]:
q4 = pd.read_sql('''
    SELECT
        PaymentMethod,
        CASE IsAutoPayment
            WHEN 1 THEN 'Auto-payment'
            ELSE        'Manual payment'
        END                                    AS payment_type,
        COUNT(*)                              AS total_customers,
        SUM(Churn)                           AS churned,
        ROUND(AVG(Churn) * 100.0, 1)        AS churn_rate_pct
    FROM customers
    GROUP BY PaymentMethod, IsAutoPayment
    ORDER BY churn_rate_pct DESC
''', conn)

print('Query 4 — Churn by Payment method:')
display(q4)
print()

# Auto vs manual comparison
q4b = pd.read_sql('''
    SELECT
        CASE IsAutoPayment
            WHEN 1 THEN 'Auto-payment'
            ELSE        'Manual payment'
        END                                    AS payment_type,
        COUNT(*)                              AS total_customers,
        SUM(Churn)                           AS churned,
        ROUND(AVG(Churn) * 100.0, 1)        AS churn_rate_pct
    FROM customers
    GROUP BY IsAutoPayment
    ORDER BY churn_rate_pct DESC
''', conn)
print('Auto vs Manual summary:')
display(q4b)
print()

elec_r  = q4[q4['PaymentMethod']=='Electronic check']['churn_rate_pct'].values[0]
elec_n  = q4[q4['PaymentMethod']=='Electronic check']['total_customers'].values[0]
auto_r  = q4b[q4b['payment_type']=='Auto-payment']['churn_rate_pct'].values[0]
manual_r= q4b[q4b['payment_type']=='Manual payment']['churn_rate_pct'].values[0]
print('Key insight:')
print(f'  Electronic check churn : {elec_r}%  ({elec_n:,} customers)')
print(f'  Auto-payment avg churn : {auto_r}%')
print(f'  Manual payment avg     : {manual_r}%')
print(f'  Manual customers churn {manual_r/auto_r:.1f}× more than auto-payment')
print()
print('Retention action: SMS/email campaign — convert Electronic check')
print('to auto-payment with €5 credit incentive. Low cost, high impact.')

Query 4 — Churn by Payment method:


,PaymentMethod,payment_type,total_customers,churned,churn_rate_pct
0,Electronic check,Manual payment,2365,1071,45.30
1,Mailed check,Manual payment,1612,308,19.10
2,Bank transfer (automatic),Auto-payment,1544,258,16.70
3,Credit card (automatic),Auto-payment,1522,232,15.20



Auto vs Manual summary:


,payment_type,total_customers,churned,churn_rate_pct
0,Manual payment,3977,1379,34.70
1,Auto-payment,3066,490,16.00



Key insight:
  Electronic check churn : 45.3%  (2,365 customers)
  Auto-payment avg churn : 16.0%
  Manual payment avg     : 34.7%
  Manual customers churn 2.2× more than auto-payment

Retention action: SMS/email campaign — convert Electronic check
to auto-payment with €5 credit incentive. Low cost, high impact.


---
### Query 5 — Financial impact of churn

**Business question:** What is the total economic cost of churn?

This query quantifies the business problem and justifies retention investment.
It computes the **capital lost** (total revenue from churned customers)
and the **annual net loss** assuming a 2% banking margin.

In [17]:
q5 = pd.read_sql('''
    SELECT
        COUNT(*)                                                     AS total_customers,
        SUM(Churn)                                                   AS churned_customers,
        COUNT(*) - SUM(Churn)                                        AS retained_customers,
        ROUND(AVG(Churn) * 100.0, 1)                                AS churn_rate_pct,

        -- Capital lost: total value generated by churned customers before leaving
        ROUND(
            SUM(CASE WHEN Churn = 1 THEN TotalCharges ELSE 0 END)
        , 2)                                                         AS capital_lost_eur,

        -- Avg lifetime value of a churned customer
        ROUND(AVG(CASE WHEN Churn = 1 THEN TotalCharges END), 2)    AS avg_ltv_churned_eur,

        -- Avg lifetime value of a retained customer
        ROUND(AVG(CASE WHEN Churn = 0 THEN TotalCharges END), 2)    AS avg_ltv_retained_eur,

        -- Annual net loss at 2% banking margin
        ROUND(
            SUM(CASE WHEN Churn = 1 THEN TotalCharges ELSE 0 END) * 0.02
        , 2)                                                         AS annual_net_loss_eur
    FROM customers
''', conn)

print('Query 5 — Financial impact of churn:')
display(q5.T.rename(columns={0: 'Value'}))
print()

cap_lost    = q5['capital_lost_eur'].values[0]
net_loss    = q5['annual_net_loss_eur'].values[0]
avg_churned = q5['avg_ltv_churned_eur'].values[0]
avg_kept    = q5['avg_ltv_retained_eur'].values[0]
print('Key insight:')
print(f'  Capital lost to churn  : €{cap_lost:,.2f}')
print(f'  Annual net loss (2%)   : €{net_loss:,.2f}')
print(f'  Avg LTV churned        : €{avg_churned:,.2f}')
print(f'  Avg LTV retained       : €{avg_kept:,.2f}')
print(f'  Retained worth         : {avg_kept/avg_churned:.1f}× more than churned')
print()
print('Every churner costs the company €1,532 in lost lifetime value.')
print('Preventing even 1 churn per day = €559,180/year saved.')

Query 5 — Financial impact of churn:


,Value
total_customers,7043.00
churned_customers,1869.00
retained_customers,5174.00
churn_rate_pct,26.50
capital_lost_eur,2862926.90
avg_ltv_churned_eur,1531.80
avg_ltv_retained_eur,2550.00
annual_net_loss_eur,57258.54



Key insight:
  Capital lost to churn  : €2,862,926.90
  Annual net loss (2%)   : €57,258.54
  Avg LTV churned        : €1,531.80
  Avg LTV retained       : €2,550.00
  Retained worth         : 1.7× more than churned

Every churner costs the company €1,532 in lost lifetime value.
Preventing even 1 churn per day = €559,180/year saved.


---
### Query 6 — High-risk customer profile

**Business question:** Who should the retention team call first?

This query finds **active customers** (Churn=0) who combine
the three highest-risk factors simultaneously:
- Month-to-month contract (42.7% churn)
- Fiber optic internet (41.9% churn)
- Electronic check payment (45.3% churn)
- Tenure ≤ 12 months (48.3% churn)

These customers have already LEFT in 47%+ of similar cases.
They are the immediate retention priority.

In [19]:
q6 = pd.read_sql('''
    SELECT
        customerID,
        tenure,
        Contract,
        InternetService,
        PaymentMethod,
        ROUND(MonthlyCharges, 2)  AS MonthlyCharges,
        NumServices
    FROM customers
    WHERE
        Contract        = 'Month-to-month'   -- 42.7% churn segment
        AND InternetService = 'Fiber optic'  -- 41.9% churn segment
        AND PaymentMethod   = 'Electronic check' -- 45.3% churn segment
        AND tenure          <= 12            -- 48.3% churn in first year
        AND Churn           = 0             -- still active — retention target
    ORDER BY MonthlyCharges DESC
    LIMIT 10
''', conn)

print('Query 6 — Top 10 highest-risk ACTIVE customers:')
print('(Criteria: Month-to-month + Fiber optic + Electronic check + tenure ≤ 12)')
display(q6)
print()

# Total count and financials
q6_stats = pd.read_sql('''
    SELECT
        COUNT(*)                          AS total_high_risk_active,
        ROUND(AVG(MonthlyCharges), 2)    AS avg_monthly_eur,
        ROUND(SUM(MonthlyCharges), 2)    AS total_monthly_revenue_eur,
        ROUND(AVG(NumServices), 1)       AS avg_services
    FROM customers
    WHERE
        Contract        = 'Month-to-month'
        AND InternetService = 'Fiber optic'
        AND PaymentMethod   = 'Electronic check'
        AND tenure          <= 12
        AND Churn           = 0
''', conn)

n_hr    = q6_stats['total_high_risk_active'].values[0]
avg_hr  = q6_stats['avg_monthly_eur'].values[0]
tot_hr  = q6_stats['total_monthly_revenue_eur'].values[0]
svc_hr  = q6_stats['avg_services'].values[0]

print(f'Total high-risk active customers  : {n_hr}')
print(f'Average monthly charge            : €{avg_hr}')
print(f'Total monthly revenue at risk     : €{tot_hr:,.2f}')
print(f'Annual revenue at risk            : €{tot_hr*12:,.2f}')
print(f'Average number of services        : {svc_hr}')
print()
print('Priority action: contact all', n_hr, 'customers immediately.')
print('Offer: One year contract + auto-payment setup + 10% discount.')
print(f'Cost of campaign  : €{n_hr * 15:,} (€15 per contact)')
print(f'Revenue preserved : €{tot_hr*12:,.0f}/year if all retained')

Query 6 — Top 10 highest-risk ACTIVE customers:
(Criteria: Month-to-month + Fiber optic + Electronic check + tenure ≤ 12)


,customerID,tenure,Contract,InternetService,PaymentMethod,MonthlyCharges,NumServices
0,3292-PBZEJ,11,Month-to-month,Fiber optic,Electronic check,111.40,8
1,2081-VEYEH,3,Month-to-month,Fiber optic,Electronic check,107.95,7
2,6734-GMPVK,5,Month-to-month,Fiber optic,Electronic check,105.30,7
3,7145-FEJWU,12,Month-to-month,Fiber optic,Electronic check,105.30,7
4,2018-PZKMU,9,Month-to-month,Fiber optic,Electronic check,103.10,6
5,8118-TJAFG,9,Month-to-month,Fiber optic,Electronic check,101.50,6
6,0722-SVSFK,7,Month-to-month,Fiber optic,Electronic check,100.40,6
7,7379-FNIUJ,2,Month-to-month,Fiber optic,Electronic check,100.20,6
8,6645-MXQJT,2,Month-to-month,Fiber optic,Electronic check,97.10,5
9,4566-NECEV,5,Month-to-month,Fiber optic,Electronic check,96.55,6



Total high-risk active customers  : 182
Average monthly charge            : €81.15
Total monthly revenue at risk     : €14,768.95
Annual revenue at risk            : €177,227.40
Average number of services        : 3.6

Priority action: contact all 182 customers immediately.
Offer: One year contract + auto-payment setup + 10% discount.
Cost of campaign  : €2,730 (€15 per contact)
Revenue preserved : €177,227/year if all retained


---
## 4 · Export query results to CSV

In [21]:
exports = {
    'sql_q1_contract_churn.csv'      : q1,
    'sql_q2_internet_churn.csv'      : q2,
    'sql_q3_tenuregroup_churn.csv'   : q3,
    'sql_q4_payment_churn.csv'       : q4,
    'sql_q5_financial_impact.csv'    : q5,
    'sql_q6_high_risk_customers.csv' : q6,
}

for filename, dataframe in exports.items():
    path = OUTPUT_DIR / filename
    dataframe.to_csv(path, index=False)
    print(f'Saved: {filename}  ({len(dataframe)} rows)')

conn.close()
print()
print(f'Database connection closed.')
print(f'SQLite file : {DB_PATH}')
print(f'Size        : {DB_PATH.stat().st_size / 1024:.1f} KB')

Saved: sql_q1_contract_churn.csv  (3 rows)
Saved: sql_q2_internet_churn.csv  (3 rows)
Saved: sql_q3_tenuregroup_churn.csv  (4 rows)
Saved: sql_q4_payment_churn.csv  (4 rows)
Saved: sql_q5_financial_impact.csv  (1 rows)
Saved: sql_q6_high_risk_customers.csv  (10 rows)

Database connection closed.
SQLite file : C:\Users\natal\Documents\churn_project\outputs\telco_churn.db
Size        : 932.0 KB


---
## Summary

| Query | Business question | Key finding |
|---|---|---|
| Q1 — Contract | Does commitment level affect churn? | Month-to-month 42.7% vs Two year 2.8% — 15× ratio |
| Q2 — Internet | Does internet type affect churn? | Fiber optic 41.9% vs No internet 7.4% — 5.7× ratio |
| Q3 — Lifecycle | When do customers churn? | New (0-12m): 47.4% — critical onboarding window |
| Q4 — Payment | Does payment method reveal engagement? | Electronic check 45.3% vs auto-pay 16% — 2.8× ratio |
| Q5 — Financial | What does churn cost? | €2.86M capital lost · €57K annual net loss at 2% |
| Q6 — High-risk | Who to call first? | 182 active customers matching all 4 risk factors |

**Generated outputs:**
```
outputs/
├── telco_churn.db                   ← SQLite database (24 columns)
├── sql_q1_contract_churn.csv
├── sql_q2_internet_churn.csv
├── sql_q3_tenuregroup_churn.csv
├── sql_q4_payment_churn.csv
├── sql_q5_financial_impact.csv
└── sql_q6_high_risk_customers.csv
```

**Next → Notebook 04 · ML Training**